In [1]:
import cv2
import mediapipe as mp
import numpy as np
from inference import get_model
import supervision as sv
from collections import deque
import os

In [2]:
print(cv2.__version__)  
print(mp.__version__)  
print(np.__version__)   
print(sv.__version__)   

4.10.0
0.10.15
1.26.4
0.25.1


In [3]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

I0000 00:00:1735745706.582282 14543004 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 83), renderer: Apple M2


In [4]:
model = get_model(model_id="ball_detection-onors/3", api_key="YOUR_ROBOFLOW_API_KEY")

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
2025-01-01 23:35:06.663524 [W:onnxruntime:, coreml_execution_provider.cc:107 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 11 number of nodes in the graph: 233 number of nodes supported by CoreML: 221
W0000 00:00:1735745706.712115 14564325 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1735745706.735032 14564318 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [5]:
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Midpoint
    c = np.array(c)  # End point

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360.0 - angle

    return angle

In [6]:
def detect_jump(landmarks):
    left_foot = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y
    right_foot = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y
    ground_level = max(left_foot, right_foot)
    return left_foot < ground_level - 0.02 and right_foot < ground_level - 0.02

In [7]:
def analyze_pose(landmarks, knee_angle_window):
    feedback = {"squared_up": False}

    # Get relevant joint coordinates
    left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                 landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
    left_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                  landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]

    # Calculate angles
    knee_angle = calculate_angle(left_hip, left_knee, left_ankle)

    # Correct for standing straight where angle should be 180
    if knee_angle < 90:  # Adjusting logic to handle standing straight
        knee_angle = 180 - knee_angle

    # Add to the rolling window
    knee_angle_window.append(knee_angle)

    # Check if the current angle is in a desirable range
    if len(knee_angle_window) >= 3 and knee_angle_window[-2] == min(list(knee_angle_window)[-3:]):
        feedback["squared_up"] = True

    return feedback

In [8]:
def detect_release(ball_detections, landmarks):
    if not ball_detections:
        return False

    left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                  landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
    right_wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                   landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

    for detection in ball_detections:
        ball_center = [(detection[0] + detection[2]) / 2, (detection[1] + detection[3]) / 2]
        if np.linalg.norm(np.array(ball_center) - np.array(left_wrist)) < 0.05 or \
           np.linalg.norm(np.array(ball_center) - np.array(right_wrist)) < 0.05:
            return False

    return True

In [10]:
video_path = "shooting.mp4"
cap = cv2.VideoCapture(video_path)

feedback_results = {"squared_up": False, "release": False}
knee_angle_window = deque(maxlen=5)  # Rolling window for knee angles
squared_up_saved = False
release_saved = False
ball_last_detected = True
squared_up_frame_counter = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run Mediapipe Pose detection
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = pose.process(rgb_frame)

    # Run YOLO for ball detection
    results = model.infer(frame)[0]
    detections = sv.Detections.from_inference(results)
    ball_detected = len(detections) > 0

    # Annotate ball detection
    bounding_box_annotator = sv.BoxAnnotator()
    annotated_frame = bounding_box_annotator.annotate(scene=frame, detections=detections)

    if result.pose_landmarks:
        landmarks = result.pose_landmarks.landmark
        feedback = analyze_pose(landmarks, knee_angle_window)

        # Aggregate feedback for squaring up
        if feedback["squared_up"]:
            feedback_results["squared_up"] = True
            if not squared_up_saved:
                cv2.imwrite("results/squared_up_frame.jpg", frame)
                squared_up_saved = True
        else:
            squared_up_frame_counter = 0

        # Detect release (ball leaving hand + jump)
        if not ball_detected and ball_last_detected and detect_jump(landmarks):
            feedback_results["release"] = True
            if not release_saved:
                cv2.imwrite("results/release_frame.jpg", frame)
                release_saved = True

    ball_last_detected = ball_detected

    # Display detections (optional)
    cv2.imshow('Frame', annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [11]:
print("Feedback Results:")
print(f"Squared Up: {'Yes' if feedback_results['squared_up'] else 'No'}")
print(f"Release: {'Yes' if feedback_results['release'] else 'No'}")

Feedback Results:
Squared Up: Yes
Release: No


In [ ]:
print(f"Knee Angle Window: {list(knee_angle_window)}")
